# Calculate the metrics from the csv

In [ ]:
import pandas as pd
import numpy as np

def calculate_metrics_from_csv(csv_path):
    # 1. Load the distance matrix
    # Using sep=';' as defined in your CSV construction
    df = pd.read_csv(csv_path, sep=';')
    
    # queryId is the first column, the rest are galleryIds
    query_ids = df['queryId'].values
    gallery_ids = df.columns[1:].values
    dist_mat = df.iloc[:, 1:].values  # Extract only the distance values
    
    # 2. Helper function to extract actual Dog ID from sample ID
    # Edit this if your IDs are formatted differently (e.g., 'dog001_v1' -> 'dog001')
    get_dog_label = lambda x: str(x).split('_')[0]
    
    num_queries = len(query_ids)
    all_aps = []
    all_cmc = []

    print(f"-> Processing {num_queries} queries from {csv_path}...")

    for i in range(num_queries):
        q_label = get_dog_label(query_ids[i])
        
        # Sort gallery indices by distance (ascending)
        sort_idx = np.argsort(dist_mat[i])
        sorted_gallery_labels = [get_dog_label(gallery_ids[j]) for j in sort_idx]
        
        # Create a boolean mask of matches
        matches = np.array([label == q_label for label in sorted_gallery_labels])
        
        # --- CMC Calculation (Rank-N) ---
        if any(matches):
            first_match_rank = np.where(matches == True)[0][0]
            cmc = np.zeros(len(matches))
            cmc[first_match_rank:] = 1
            all_cmc.append(cmc)
        
        # --- AP Calculation (mAP) ---
        num_rel = np.sum(matches)
        if num_rel > 0:
            # Cumulative matches: [0, 0, 1, 1, 2...]
            cum_matches = np.cumsum(matches)
            # Precision at each rank: [0, 0, 1/3, 2/4...]
            precisions = cum_matches / (np.arange(len(matches)) + 1)
            # AP = sum(precision at each correct rank) / total relevant
            ap = np.sum(precisions * matches) / num_rel
            all_aps.append(ap)

    # 3. Final Results summary
    mAP = np.mean(all_aps)
    cmc_avg = np.mean(all_cmc, axis=0)
    
    print("\n" + "="*40)
    print("📈 PRECOMPUTED DISTANCE METRICS")
    print("-"*40)
    print(f"mAP      : {mAP:.2%}")
    print(f"Rank-1   : {cmc_avg[0]:.2%}")
    print(f"Rank-5   : {cmc_avg[4]:.2%}")
    print(f"Rank-10  : {cmc_avg[9]:.2%}")
    print("="*40 + "\n")

    return {"mAP": mAP, "Rank-1": cmc_avg[0]}

# To run:
# calculate_metrics_from_csv("experiments/dinov2_closed_v1/dist_matrix.csv")